In [2]:
import joblib
import numpy as np
import pandas as pd

# Load model + decision threshold
model = joblib.load("calibrated_rf.pkl")
threshold = joblib.load("best_cost_threshold.pkl")

# Load test set for simulation (or any batch data)
X_test = joblib.load("X_test.pkl")
y_test = joblib.load("y_test.pkl")

threshold


np.float64(0.1653061224489796)

In [3]:
def predict_survival(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    input_df: DataFrame with the same feature columns used in training.
    returns: Probability + Prediction (based on cost-optimal threshold)
    """
    proba = model.predict_proba(input_df)[:, 1]
    pred = (proba >= threshold).astype(int)

    out = pd.DataFrame({
        "Probability_survived": proba,
        "Prediction_survived": pred
    }, index=input_df.index)

    return out


In [4]:
results = predict_survival(X_test)

results.head()


,Probability_survived,Prediction_survived
1148,0.092712,0
1049,0.198996,1
982,0.092511,0
808,0.109642,0
1195,0.334890,1


In [5]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_deploy = results["Prediction_survived"].values

print("Threshold used:", threshold)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_deploy))
print("\nClassification report:\n", classification_report(y_test, y_pred_deploy))


Threshold used: 0.1653061224489796
Confusion matrix:
 [[121  68]
 [ 11  62]]

Classification report:
               precision    recall  f1-score   support

           0       0.92      0.64      0.75       189
           1       0.48      0.85      0.61        73

    accuracy                           0.70       262
   macro avg       0.70      0.74      0.68       262
weighted avg       0.79      0.70      0.71       262



In [6]:
sample_idx = X_test.index[:5]
demo = X_test.loc[sample_idx].copy()

demo_out = predict_survival(demo)
pd.concat([demo, demo_out], axis=1)


,Pclass,Sex,Age,Fare_log,FamilySize,IsAlone,Probability_survived,Prediction_survived
1148,3,0,28.0,2.202765,1,1,0.092712,0
1049,1,0,42.0,3.316003,1,1,0.198996,1
982,3,0,28.0,2.171907,1,1,0.092511,0
808,2,0,39.0,2.639057,1,1,0.109642,0
1195,3,1,28.0,2.169054,1,1,0.334890,1


In [7]:
new_passenger = X_test.iloc[[0]].copy()

# Örnek: bazı alanları değiştir (kolon adların sende aynı olmalı)
if "Age" in new_passenger.columns:
    new_passenger["Age"] = 25
if "Pclass" in new_passenger.columns:
    new_passenger["Pclass"] = 1
if "Sex" in new_passenger.columns:
    # senin encoding’ine göre: female=1 / male=0 veya tersi olabilir
    # burada sadece örnek olsun diye 1 verdim
    new_passenger["Sex"] = 1
if "IsAlone" in new_passenger.columns:
    new_passenger["IsAlone"] = 0
if "FamilySize" in new_passenger.columns:
    new_passenger["FamilySize"] = 2
if "Fare_log" in new_passenger.columns:
    new_passenger["Fare_log"] = 2.4


predict_survival(new_passenger)


,Probability_survived,Prediction_survived
1148,0.53053,1


In [8]:
package = {
    "model": model,
    "threshold": float(threshold),
    "meta": {
        "policy": "cost_optimal_threshold",
        "notes": "Calibrated RF + cost-based threshold for decision rule"
    }
}

joblib.dump(package, "deployment_package.pkl")


['deployment_package.pkl']

In [9]:
# verifying thet the package  works
pkg = joblib.load("deployment_package.pkl")

model2 = pkg["model"]
thr2 = pkg["threshold"]

def predict_survival_pkg(input_df: pd.DataFrame) -> pd.DataFrame:
    proba = model2.predict_proba(input_df)[:, 1]
    pred = (proba >= thr2).astype(int)
    return pd.DataFrame({"Probability_survived": proba, "Prediction_survived": pred}, index=input_df.index)

predict_survival_pkg(X_test.iloc[:3])


,Probability_survived,Prediction_survived
1148,0.092712,0
1049,0.198996,1
982,0.092511,0


### A reusable inference pipeline was implemented using the calibrated Random Forest model and the cost-optimal decision threshold. The deployment function outputs both probabilities and final decisions, and the artifacts were packaged into a single deployable file to ensure reproducibility and consistent decision-making.